In [1]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [2]:
# Load the data
import pandas as pd

df_raw = pd.read_excel("/content/2M4AFN01_v2.xlsx", sheet_name=0, header=None)

years = df_raw.iloc[2, 2:].ffill().tolist()
months = df_raw.iloc[3, 2:].tolist()
values = df_raw.iloc[4, 2:].tolist()

records = []
for y, m, v in zip(years, months, values):
    if v == '..' or pd.isna(v):
        continue
    records.append({"year": int(y), "month": m, "price": float(v)})

df = pd.DataFrame(records)
month_map = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
             "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["month_num"] = df["month"].map(month_map)
df["date"] = pd.to_datetime(dict(year=df.year, month=df.month_num, day=1))
df = df[["date", "price"]].sort_values("date").reset_index(drop=True)

print(df.shape)
print(df.head())
print(df.tail())
print(df.isna().sum())

(126, 2)
        date  price
0 2016-01-01  12.42
1 2016-02-01  16.36
2 2016-03-01  14.62
3 2016-04-01  16.59
4 2016-05-01  13.67
          date  price
121 2026-02-01  21.66
122 2026-03-01  24.31
123 2026-04-01  21.88
124 2026-05-01  19.17
125 2026-06-01  18.51
date     0
price    0
dtype: int64


In [3]:
# Split into train and test
train_df = df[df["date"] < "2026-01-01"].reset_index(drop=True)
test_df = df[df["date"] >= "2026-01-01"].reset_index(drop=True)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("overlap:", set(train_df.date) & set(test_df.date))

train shape: (120, 2)
test shape: (6, 2)
overlap: set()


In [4]:
# Component 1: 10-seed committee, season+NFA+momentum+mean_reversion, 40/60 LSTM:GRU blend
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Input, Concatenate
from sklearn.preprocessing import MinMaxScaler

def nfa_flag(d):
    return 1 if (pd.Timestamp("2019-01-01")<=d<=pd.Timestamp("2020-12-01")) or (pd.Timestamp("2025-01-01")<=d<=pd.Timestamp("2026-06-01")) else 0

# Compute every feature on the FULL combined series BEFORE splitting, so the
# 12-month rolling average for early 2026 can correctly see back into 2025's
# real prices, instead of being computed separately per split (which was the bug)
df["month_num"] = df["date"].dt.month
df["nfa_active"] = df["date"].apply(nfa_flag)
df["month_sin"] = np.sin(2*np.pi*df["month_num"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month_num"]/12)
df["momentum_3mo"] = df["price"].shift(1).pct_change(periods=3).fillna(0)
df["ma_12mo"] = df["price"].shift(1).rolling(12).mean()
df["mean_reversion"] = ((df["price"].shift(1) - df["ma_12mo"]) / df["ma_12mo"]).fillna(0)

train_df = df[df["date"] < "2026-01-01"].reset_index(drop=True)
test_df = df[df["date"] >= "2026-01-01"].reset_index(drop=True)

FEAT_COLS = ["month_sin", "month_cos", "nfa_active", "momentum_3mo", "mean_reversion"]
LOOKBACK = 6
LSTM_WEIGHT = 0.4
GRU_WEIGHT = 0.6
SEEDS = [42, 7, 123, 2024, 99, 2019, 2025, 1, 88, 555]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[["price"]])

def make_seq_with_features(prices_scaled, feat_df, lookback):
    X_price, X_feat, y = [], [], []
    for i in range(len(prices_scaled) - lookback):
        X_price.append(prices_scaled[i:i+lookback, 0])
        X_feat.append(feat_df.iloc[i+lookback][FEAT_COLS].values)
        y.append(prices_scaled[i+lookback, 0])
    return np.array(X_price), np.array(X_feat), np.array(y)

X_price_train, X_feat_train, y_train = make_seq_with_features(train_scaled, train_df, LOOKBACK)
X_price_train = X_price_train.reshape((X_price_train.shape[0], LOOKBACK, 1))
X_feat_train = X_feat_train.astype(float)
print("X_price_train shape:", X_price_train.shape, "X_feat_train shape:", X_feat_train.shape)

def build_model(rnn_layer):
    price_in = Input(shape=(LOOKBACK,1))
    feat_in = Input(shape=(len(FEAT_COLS),))
    r = rnn_layer(32, activation="tanh")(price_in)
    merged = Concatenate()([r, feat_in])
    out = Dense(16, activation="relu")(merged)
    out = Dense(1)(out)
    m = Model([price_in, feat_in], out)
    m.compile(optimizer="adam", loss="mse")
    return m

lstm_models, gru_models = [], []
for seed in SEEDS:
    tf.random.set_seed(seed); np.random.seed(seed)
    lstm_m = build_model(LSTM)
    lstm_m.fit([X_price_train, X_feat_train], y_train, epochs=80, batch_size=8, verbose=0)
    lstm_models.append(lstm_m)

    gru_m = build_model(GRU)
    gru_m.fit([X_price_train, X_feat_train], y_train, epochs=80, batch_size=8, verbose=0)
    gru_models.append(gru_m)

print(f"Trained {len(SEEDS)} independent LSTM/GRU pairs.")

X_price_train shape: (114, 6, 1) X_feat_train shape: (114, 5)
Trained 10 independent LSTM/GRU pairs.


In [5]:
# Test on the 2026 months
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

full_df = pd.concat([train_df, test_df])[["date","price","month_sin","month_cos","nfa_active","momentum_3mo","mean_reversion"]].reset_index(drop=True)
full_scaled = scaler.transform(full_df[["price"]])
n_train = len(train_df)

X_price_test, X_feat_test, _ = make_seq_with_features(full_scaled, full_df, LOOKBACK)
start = n_train - LOOKBACK
X_price_test_slice = X_price_test[start:start+len(test_df)].reshape((len(test_df), LOOKBACK, 1))
X_feat_test_slice = X_feat_test[start:start+len(test_df)].astype(float)
actual = test_df["price"].values

lstm_preds = [scaler.inverse_transform(m.predict([X_price_test_slice, X_feat_test_slice], verbose=0)).flatten() for m in lstm_models]
gru_preds = [scaler.inverse_transform(m.predict([X_price_test_slice, X_feat_test_slice], verbose=0)).flatten() for m in gru_models]
lstm_pred = np.mean(lstm_preds, axis=0)
gru_pred = np.mean(gru_preds, axis=0)
ensemble_pred = LSTM_WEIGHT * lstm_pred + GRU_WEIGHT * gru_pred

results = pd.DataFrame({"date": test_df["date"], "actual": actual, "ensemble_pred": ensemble_pred})
print(results)
r2 = r2_score(actual, ensemble_pred); mae = mean_absolute_error(actual, ensemble_pred)
rmse = np.sqrt(mean_squared_error(actual, ensemble_pred))
mape = np.mean(np.abs((actual - ensemble_pred) / actual)) * 100
print(f"Ensemble: R2={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%  Accuracy={100-mape:.2f}%")

naive_pred = np.array([full_df.set_index("date")["price"].loc[d - pd.DateOffset(months=1)] for d in test_df["date"]])
print(f"Naive baseline: R2={r2_score(actual, naive_pred):.4f}  MAE={mean_absolute_error(actual, naive_pred):.4f}")

        date  actual  ensemble_pred
0 2026-01-01   20.25      15.484343
1 2026-02-01   21.66      19.837355
2 2026-03-01   24.31      21.622860
3 2026-04-01   21.88      23.479973
4 2026-05-01   19.17      21.699051
5 2026-06-01   18.51      19.532871
Ensemble: R2=-0.9424  MAE=2.4046  RMSE=2.6850  MAPE=11.51%  Accuracy=88.49%
Naive baseline: R2=-1.5769  MAE=2.6250


In [6]:
# Component 2: Random Forest & SVR (kept blind to season/NFA - raw price only,
# so it remains an independent second opinion for anomaly detection)

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

def make_tab(pdf, lookback):
    X, y = [], []
    for i in range(len(pdf) - lookback):
        window_price = pdf["price"].values[i:i+lookback]
        X.append(window_price)
        y.append(pdf["price"].values[i+lookback])
    return np.array(X), np.array(y)

X_train_tab, y_train_tab = make_tab(train_df, LOOKBACK)
print("X_train_tab shape:", X_train_tab.shape)   # expect (114, 6) now, not (114, 9)

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train_tab, y_train_tab)

svr_scaler = MinMaxScaler()
X_train_tab_s = svr_scaler.fit_transform(X_train_tab)
y_scaler_svr = MinMaxScaler()
y_train_tab_s = y_scaler_svr.fit_transform(y_train_tab.reshape(-1,1)).flatten()

svr_model = SVR(kernel="rbf", C=10, epsilon=0.01)
svr_model.fit(X_train_tab_s, y_train_tab_s)

full_df_tab = pd.concat([train_df, test_df]).reset_index(drop=True)
X_full_tab, _ = make_tab(full_df_tab, LOOKBACK)
n_train = len(train_df)
X_test_tab = X_full_tab[n_train-LOOKBACK:n_train-LOOKBACK+len(test_df)]

rf_pred = rf_model.predict(X_test_tab)
X_test_tab_s = svr_scaler.transform(X_test_tab)
svr_pred = y_scaler_svr.inverse_transform(svr_model.predict(X_test_tab_s).reshape(-1,1)).flatten()

rf_svr_results = pd.DataFrame({"date": test_df["date"], "actual": actual,
                                 "rf_pred": rf_pred, "svr_pred": svr_pred})
print(rf_svr_results)

for name, pred in [("Random Forest", rf_pred), ("SVR", svr_pred)]:
    r2 = r2_score(actual, pred)
    mae = mean_absolute_error(actual, pred)
    print(f"{name}: R2={r2:.4f}  MAE={mae:.4f}")

X_train_tab shape: (114, 6)
        date  actual   rf_pred   svr_pred
0 2026-01-01   20.25  15.11140  12.303655
1 2026-02-01   21.66  18.52225  14.385546
2 2026-03-01   24.31  20.34380  14.947913
3 2026-04-01   21.88  22.54110  18.179144
4 2026-05-01   19.17  21.91785  17.900460
5 2026-06-01   18.51  18.87730  20.157322
Random Forest: R2=-1.6989  MAE=2.6698
SVR: R2=-8.9568  MAE=5.2001


In [7]:
# 95th Percentile Threshold
import warnings
import logging

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
tf.get_logger().setLevel(logging.ERROR)  # stop TF's harmless retracing spam from printing

def compute_deviation(idx_range, all_prices, all_dates, feat_df):
    # This checks how much Component 1 (LSTM/GRU) and Component 2 (RF/SVR)
    # disagree with each other for every month in idx_range. Big disagreement
    # = possible anomaly. We batch all the predictions together instead of
    # looping one month at a time, way faster and avoids TF's retracing warning.
    idx_range = list(idx_range)

    # grab the 6-month price window and the season/NFA features for every month at once
    windows_raw = np.array([all_prices[i-LOOKBACK:i] for i in idx_range])
    feats = np.array([feat_df.iloc[i][["month_sin", "month_cos", "nfa_active", "momentum_3mo", "mean_reversion"]].values
               for i in idx_range]).astype(float)

    # scaler expects a "price" column name since that's what it was trained on
    windows_scaled = scaler.transform(
        pd.DataFrame(windows_raw.reshape(-1, 1), columns=["price"])
    ).reshape(-1, LOOKBACK, 1)

    # Component 1's guess: average each committee (5 LSTMs, 5 GRUs), then blend
    lstm_preds_all = [scaler.inverse_transform(m.predict([windows_scaled, feats], verbose=0)).flatten() for m in lstm_models]
    gru_preds_all = [scaler.inverse_transform(m.predict([windows_scaled, feats], verbose=0)).flatten() for m in gru_models]
    lstm_p = np.mean(lstm_preds_all, axis=0)
    gru_p = np.mean(gru_preds_all, axis=0)
    ensemble_p = LSTM_WEIGHT * lstm_p + GRU_WEIGHT * gru_p # was: (lstm_p + gru_p) / 2

    # Component 2's guess (average of RF and SVR) - kept blind to season/NFA on purpose,
    # so it stays an independent second opinion instead of agreeing with Component 1 by default
    rf_p = rf_model.predict(windows_raw)
    svr_p = y_scaler_svr.inverse_transform(
        svr_model.predict(svr_scaler.transform(windows_raw)).reshape(-1, 1)
    ).flatten()
    anomaly_p = (rf_p + svr_p) / 2

    # how far apart the two guesses are, as a percentage
    deviation_pct = np.abs(ensemble_p - anomaly_p) / anomaly_p * 100

    return pd.DataFrame({
        "date": [all_dates[i] for i in idx_range],
        "actual": [all_prices[i] for i in idx_range],
        "deviation_pct": deviation_pct
    })

all_prices = full_df["price"].values
all_dates = full_df["date"].values
n = len(all_prices)

full_dev_df = compute_deviation(range(LOOKBACK, n), all_prices, all_dates, full_df)

# only use the training months to decide what "normal" disagreement looks like,
# so the threshold isn't accidentally influenced by the 2026 test data or the shock itself
train_dev_df = full_dev_df[full_dev_df["date"] < train_df["date"].iloc[-1]]
ANOMALY_THRESHOLD = np.percentile(train_dev_df["deviation_pct"], 95)
print(f"Anomaly threshold (95th percentile of training deviation): {ANOMALY_THRESHOLD:.2f}%")

full_dev_df["flagged"] = full_dev_df["deviation_pct"] > ANOMALY_THRESHOLD

shock_check = full_dev_df[(full_dev_df.date >= "2025-05-01") & (full_dev_df.date <= "2025-09-01")]
print("\n2025 shock window check:")
print(shock_check.to_string(index=False))
print(f"\nShock months flagged: {shock_check['flagged'].sum()} / {len(shock_check)}")
print(f"Total months flagged in full history: {full_dev_df['flagged'].sum()} / {len(full_dev_df)}")

Anomaly threshold (95th percentile of training deviation): 8.47%

2025 shock window check:
      date  actual  deviation_pct  flagged
2025-05-01   15.18       5.714356    False
2025-06-01   11.98       1.461091    False
2025-07-01   11.18       8.499564     True
2025-08-01   14.32       6.612803    False
2025-09-01   16.43      13.252887     True

Shock months flagged: 2 / 5
Total months flagged in full history: 11 / 120


In [8]:
# Saving the models
import os, joblib, json

os.makedirs("models_output", exist_ok=True)

for i, m in enumerate(lstm_models):
    m.save(f"models_output/lstm_model_{i}.keras")
for i, m in enumerate(gru_models):
    m.save(f"models_output/gru_model_{i}.keras")

joblib.dump(rf_model, "models_output/rf_model.pkl")
joblib.dump(svr_model, "models_output/svr_model.pkl")
joblib.dump(scaler, "models_output/price_scaler.pkl")
joblib.dump(svr_scaler, "models_output/svr_input_scaler.pkl")
joblib.dump(y_scaler_svr, "models_output/svr_output_scaler.pkl")
with open("models_output/anomaly_config.json", "w") as f:
    json.dump({"threshold_pct": float(ANOMALY_THRESHOLD), "lookback": LOOKBACK}, f)

print(f"Saved {len(lstm_models)} LSTM models, {len(gru_models)} GRU models, plus Component 2 and configs.")

Saved 10 LSTM models, 10 GRU models, plus Component 2 and configs.


In [9]:
import shutil
shutil.make_archive("models_output", "zip", "models_output")

from google.colab import files
files.download("models_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>